# strøm_output

In [25]:
navne_med_farver = [
    {"navn": "Anna", "farve": "Rød"},
    {"navn": "Benjamin", "farve": "Blå"},
    {"navn": "Clara", "farve": "Grøn"},
    {"navn": "Daniel", "farve": "Grå"},
    {"navn": "Emma", "farve": "Lilla"},
    {"navn": "Bus 201", "farve": "Gul"},
]

for person in navne_med_farver:
    print(f"{person['navn']}: {person['farve']}")

Anna: Rød
Benjamin: Blå
Clara: Grøn
Daniel: Grå
Emma: Lilla
Bus 201: Gul


In [27]:
from anymap_ts import Map
import random
import time
from IPython.display import display

# Givet som (lat, lng)
LAT = 55.65777549836654
LNG = 12.105861370948363

# anymap_ts bruger (lng, lat)
m = Map(center=(LNG, LAT), zoom=18, pitch=58, bearing=25, height="600px", width="100%")
m.add_basemap("OpenStreetMap.Mapnik")
m.add_3d_buildings()
m.add_marker(LNG, LAT, name="punkt", color="#1E88E5", popup="Valgt koordinat")

random.seed()
personer = [p for p in navne_med_farver if not p["navn"].lower().startswith("bus")]

# Hjørner (h) for området personer må bevæge sig indenfor (input: lat, lng -> [lng, lat])
h1 = [12.106759008261267, 55.65861989708783]
h2 = [12.106816349671297, 55.657810931391566]
h3 = [12.104950909565892, 55.65782359387148]
h4 = [12.105068108217221, 55.65857206333853]
human_area = [h1, h2, h3, h4]

# Bus 201-rute (input er lat/lng -> konverteret til [lng, lat])
p1 = [12.107061036174597, 55.65773066383307]
p2 = [12.105830262981327, 55.65777189215283]
p3 = [12.10402256017052, 55.657776213658096]
p4 = [12.104235395707752, 55.659645821205125]
p5 = [12.107563231968914, 55.65970153877991]
p6 = [12.107958394657734, 55.658674908889644]
p7 = [12.107126427302157, 55.65773627741259]

# Hastighedsoptimering: spring direkte fra p4 til p5 (kan sættes til False)
TELEPORT_P4_P5 = True

# Tegn kun hvis markøren faktisk flytter sig synligt
_DRAW_EPS = 1e-7
_last_drawn = {}

def _should_redraw(marker_id, lng, lat, on_bus_flag=None):
    prev = _last_drawn.get(marker_id)
    if prev is None:
        _last_drawn[marker_id] = (lng, lat, on_bus_flag)
        return True
    prev_lng, prev_lat, prev_flag = prev
    moved = (
        abs(prev_lng - lng) > _DRAW_EPS
        or abs(prev_lat - lat) > _DRAW_EPS
        or prev_flag != on_bus_flag
    )
    if moved:
        _last_drawn[marker_id] = (lng, lat, on_bus_flag)
    return moved

def point_in_polygon(lng, lat, polygon):
    inside = False
    j = len(polygon) - 1
    for i in range(len(polygon)):
        xi, yi = polygon[i]
        xj, yj = polygon[j]
        intersects = ((yi > lat) != (yj > lat)) and (
            lng < (xj - xi) * (lat - yi) / ((yj - yi) + 1e-12) + xi
        )
        if intersects:
            inside = not inside
        j = i
    return inside

def random_point_in_polygon(polygon):
    lng_values = [p[0] for p in polygon]
    lat_values = [p[1] for p in polygon]
    min_lng, max_lng = min(lng_values), max(lng_values)
    min_lat, max_lat = min(lat_values), max(lat_values)
    while True:
        candidate_lng = random.uniform(min_lng, max_lng)
        candidate_lat = random.uniform(min_lat, max_lat)
        if point_in_polygon(candidate_lng, candidate_lat, polygon):
            return [candidate_lng, candidate_lat]

def person_color(on_bus):
    return "rgba(67,160,71,0.45)" if on_bus else "#43A047"

def move_person_marker(marker_id, lng, lat, navn, farve, on_bus=False):
    if not _should_redraw(marker_id, lng, lat, on_bus):
        return
    m.remove_marker(marker_id)
    m.add_marker(
        lng,
        lat,
        name=marker_id,
        color=person_color(on_bus),
        popup=f"{navn} ({farve})",
    )

# Opret én markør pr. person og gem position
person_positions = {}
for person in personer:
    marker_id = f"person-{person['navn']}"
    lng, lat = random_point_in_polygon(human_area)
    person_positions[marker_id] = {
        "lng": lng,
        "lat": lat,
        "navn": person["navn"],
        "farve": person["farve"],
        "on_bus": False,
        "status": "free",
        "unboard_target": None,
        "unboard_progress": 0.0,
        "unboard_step": 0.08,
    }
    move_person_marker(marker_id, lng, lat, person["navn"], person["farve"], on_bus=False)

def tick_people(step_size=0.00003, locked_ids=None):
    if locked_ids is None:
        locked_ids = set()

    for marker_id, info in person_positions.items():
        if marker_id in locked_ids:
            continue

        # Folk, der er ved at stige af, bevæger sig mod deres mål uden at stoppe bussen
        if info["status"] == "unboarding":
            t = min(1.0, info["unboard_progress"] + info["unboard_step"])
            target_lng, target_lat = info["unboard_target"]
            start_lng, start_lat = info["unboard_start"]
            next_lng = start_lng + (target_lng - start_lng) * t
            next_lat = start_lat + (target_lat - start_lat) * t
            info["lng"] = next_lng
            info["lat"] = next_lat
            info["unboard_progress"] = t
            move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)
            if t >= 1.0:
                info["status"] = "free"
                info["on_bus"] = False
                info["unboard_target"] = None
            continue

        if info["on_bus"] or info["status"] != "free":
            continue

        current_lng = info["lng"]
        current_lat = info["lat"]
        next_lng = current_lng + random.uniform(-step_size, step_size)
        next_lat = current_lat + random.uniform(-step_size, step_size)

        for _ in range(6):
            if point_in_polygon(next_lng, next_lat, human_area):
                break
            next_lng = current_lng + random.uniform(-step_size, step_size)
            next_lat = current_lat + random.uniform(-step_size, step_size)
        else:
            next_lng, next_lat = random_point_in_polygon(human_area)

        info["lng"] = next_lng
        info["lat"] = next_lat
        move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)

# Opret én bus-markør, som flyttes rundt (ingen efterladte punkter)
bus_position = {"lng": p1[0], "lat": p1[1]}
m.add_marker(p1[0], p1[1], name="bus-201", color="#FDD835", popup="Bus 201")
_last_drawn["bus-201"] = (p1[0], p1[1], None)

def move_bus(lng, lat):
    bus_position["lng"] = lng
    bus_position["lat"] = lat

    if _should_redraw("bus-201", lng, lat, None):
        m.remove_marker("bus-201")
        m.add_marker(
            lng,
            lat,
            name="bus-201",
            color="#FDD835",
            popup="Bus 201",
        )

    # Ombord-personer følger bussen 1:1
    for marker_id, info in person_positions.items():
        if not info["on_bus"]:
            continue
        info["lng"] = lng
        info["lat"] = lat
        move_person_marker(marker_id, lng, lat, info["navn"], info["farve"], on_bus=True)

def bus_is_at_stop(stop_point, tolerance=0.00001):
    stop_lng, stop_lat = stop_point
    return (
        abs(bus_position["lng"] - stop_lng) <= tolerance
        and abs(bus_position["lat"] - stop_lat) <= tolerance
    )

def board_people_at_p2(max_board=3, board_seconds=2.0, board_steps=20):
    candidates = [mid for mid, info in person_positions.items() if not info["on_bus"] and info["status"] == "free"]
    if not candidates:
        return

    selected = random.sample(candidates, k=min(max_board, len(candidates)))
    starts = {mid: (person_positions[mid]["lng"], person_positions[mid]["lat"]) for mid in selected}

    # Led de valgte personer hen til bus-positionen med stabil interpolation
    for step in range(1, board_steps + 1):
        t = step / board_steps
        target_lng = bus_position["lng"]
        target_lat = bus_position["lat"]

        for marker_id in selected:
            info = person_positions[marker_id]
            start_lng, start_lat = starts[marker_id]
            next_lng = start_lng + (target_lng - start_lng) * t
            next_lat = start_lat + (target_lat - start_lat) * t
            info["lng"] = next_lng
            info["lat"] = next_lat
            move_person_marker(marker_id, next_lng, next_lat, info["navn"], info["farve"], on_bus=False)

        tick_people(locked_ids=set(selected))
        time.sleep(board_seconds / board_steps)

    for marker_id in selected:
        info = person_positions[marker_id]
        info["on_bus"] = True
        info["status"] = "on_bus"
        info["lng"] = bus_position["lng"]
        info["lat"] = bus_position["lat"]
        move_person_marker(marker_id, bus_position["lng"], bus_position["lat"], info["navn"], info["farve"], on_bus=True)

def start_unboard_at_p4():
    riders = [mid for mid, info in person_positions.items() if info["on_bus"]]
    if not riders:
        return

    for marker_id in riders:
        info = person_positions[marker_id]
        target = random_point_in_polygon(human_area)
        info["status"] = "unboarding"
        info["on_bus"] = False
        info["unboard_target"] = target
        info["unboard_start"] = (bus_position["lng"], bus_position["lat"])
        info["unboard_progress"] = 0.0

def move_segment(start, end, seconds=6, steps=60):
    frame_dt = seconds / steps
    next_ts = time.perf_counter()

    for i in range(1, steps + 1):
        t = i / steps
        lng = start[0] + (end[0] - start[0]) * t
        lat = start[1] + (end[1] - start[1]) * t
        move_bus(lng, lat)
        tick_people()

        next_ts += frame_dt
        sleep_s = next_ts - time.perf_counter()
        if sleep_s > 0:
            time.sleep(sleep_s)

def teleport_bus(target):
    move_bus(target[0], target[1])
    tick_people()

def pause_at_p2_with_boarding(total_seconds=10, interval=0.25):
    if bus_is_at_stop(p2):
        board_people_at_p2(max_board=3, board_seconds=2.0, board_steps=20)
        remaining = max(0.0, total_seconds - 2.0)
    else:
        remaining = total_seconds

    ticks = max(1, int(remaining / interval))
    for _ in range(ticks):
        tick_people()
        time.sleep(interval)

display(m)

# Uendeligt loop: p1 -> p2 (pause + boarding) -> p3 -> p4 (afstigning) -> p5 -> p6 -> p7 -> p1
while True:
    move_segment(p1, p2, seconds=6, steps=60)
    pause_at_p2_with_boarding(total_seconds=10, interval=0.25)
    move_segment(p2, p3, seconds=6, steps=60)
    move_segment(p3, p4, seconds=6, steps=60)
    if bus_is_at_stop(p4):
        start_unboard_at_p4()

    if TELEPORT_P4_P5:
        teleport_bus(p5)
    else:
        move_segment(p4, p5, seconds=6, steps=60)

    move_segment(p5, p6, seconds=6, steps=60)
    move_segment(p6, p7, seconds=6, steps=60)
    move_segment(p7, p1, seconds=6, steps=60)
    # Ingen pause ved p1 - næste tur starter med det samme

KeyboardInterrupt: 